
#ФиКЛ, Домашнее задание 4 (бонусное), ГО

## Общая информация

**Максимальная оценка: 4 балла**

Задание выполняется самостоятельно. Похожие решения считаются плагиатом — все участники получают **0 баллов**. Если нашли решение в открытом источнике — укажите ссылку.

Использование LLM допустимо: не более **30%** кода, укажите модель и промпт, опишите опыт в конце работы.

---

## О задании

Мы исследуем новости РИА и комментарии ВКонтакте. Три части:

1. **Классификация новостей** — обучить нейросеть, которая по заголовку предсказывает тег
2. **Сентимент-анализ** — запустить готовый HuggingFace-классификатор на комментариях
3. **Аналитика** — найти самые позитивные новости

| Часть | Баллы |
|-------|-------|
| Часть 1: классификация | 2 |
| Часть 2: сентимент-анализ | 1 |
| Часть 3: аналитика | 1 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from collections import Counter
import re

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
import gdown
url = "https://drive.google.com/drive/folders/11oCcLplWtp_qm-WuEbfCFP_Mz5K_z3ps?usp=sharing"
gdown.download_folder(url, quiet=True, use_cookies=False)

## Описание данных

**`ria_news.tsv`** — новости РИА (март–декабрь 2018): `href`, `date`, `title`, `snippet`, `text`, `tags`

**`vk_news.tsv`** — посты РИА ВКонтакте: `href`, `title`, `likes`, `comments`

**`vk_comments.tsv`** — комментарии к постам: `post_id`, `text`, `likes`

In [ ]:
df_ria = pd.read_csv('news_data/ria_news.tsv', sep='\t')
df_ria = df_ria[~df_ria.tags.isnull()]
print('RIA:', df_ria.shape)
df_ria.head(3)

In [ ]:
df_vk = pd.read_csv('news_data/vk_news.tsv', sep='\t')
df_vk['snippet'] = df_vk['text']
df_vk.drop('text', axis=1, inplace=True)
print('VK news:', df_vk.shape)
df_vk.head(3)

In [ ]:
df_comments = pd.read_csv('news_data/vk_comments.tsv', sep='\t', low_memory=False)
df_comments = df_comments[~df_comments.text.isnull()]
print('Comments:', df_comments.shape)
df_comments.head(3)

---

# Часть 1. Классификация новостей по тегам (5 баллов)

Задача: по заголовку новости предсказать её тематический тег.

## 1.1 Подготовка данных

Этот блок запустите как есть — он подготавливает теги, словарь и разбивает выборку по времени.

In [ ]:
# --- Теги ---
df_ria['tags'] = df_ria.tags.apply(
    lambda w: ','.join([item.strip() for item in w.lower().split(',')])
)
tags_cnt = Counter(','.join(df_ria.tags.values).split(','))
target_tags = {tag for tag, cnt in tags_cnt.most_common() if cnt > 30}
tag2idx = dict(zip(target_tags, range(len(target_tags))))
idx2tag = {v: k for k, v in tag2idx.items()}
CLASSES_NUM = len(idx2tag)

df_ria['target_tags'] = df_ria.tags.apply(
    lambda w: [tag2idx[t] for t in w.split(',') if t in target_tags]
)
df_ria = df_ria[df_ria.target_tags.apply(len) > 0]

# --- Предобработка текста ---
stops_ru = set(stopwords.words('russian'))

def normalise_text(text):
    text = text.lower()
    text = re.sub('[^а-яa-z0-9 ]', '', text)
    return text.strip()

df_ria['title_clean'] = df_ria.title.apply(normalise_text)

# --- Словарь ---
VOCAB_SIZE = 10000

def create_vocab(text):
    word_cnt = Counter(word_tokenize(text))
    vocab = {'#PAD#': 0, '#UNK#': 1}
    k = 2
    for word, _ in word_cnt.most_common():
        if word not in stops_ru:
            vocab[word] = k
            k += 1
    return vocab

vocabulary = create_vocab(' '.join(df_ria.title_clean.values))

# --- Разбивка по времени ---
# Убираем пересечение с ВК (оно пойдёт в отложенную выборку)
test_hrefs = set(df_vk.href.values) & set(df_ria.href.values)
df = df_ria[~df_ria.href.isin(test_hrefs)].copy()

df_test  = df[df.date >= '2018-12-01']
df_val   = df[(df.date >= '2018-10-01') & (df.date < '2018-12-01')]
df_train = df[df.date < '2018-10-01']

print(f'Тегов: {CLASSES_NUM}, Словарь: {len(vocabulary)}')
print(f'Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}')

In [ ]:
# Датасет
MAX_TITLE_LEN = 20

class NewsDataset(Dataset):
    def __init__(self, target, title, vocab, vocab_size, max_len, n_classes):
        self.vocab = {w: i for w, i in vocab.items() if i < vocab_size}
        self.y = self._ohe(target, n_classes)
        self.X = self._encode(title, max_len)

    def _ohe(self, target, n_classes):
        y = torch.zeros(len(target), n_classes)
        for i, tags in enumerate(target):
            y[i, tags] = 1.0
        return y

    def _encode(self, texts, max_len):
        result = []
        for sent in texts:
            tokens = [self.vocab.get(w, 1) for w in word_tokenize(sent)]
            tokens = tokens[:max_len] + [0] * max(0, max_len - len(tokens))
            result.append(tokens)
        return torch.tensor(result, dtype=torch.long)

    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


train_dataset = NewsDataset(df_train.target_tags.values, df_train.title_clean.values,
                            vocabulary, VOCAB_SIZE, MAX_TITLE_LEN, CLASSES_NUM)
val_dataset   = NewsDataset(df_val.target_tags.values,   df_val.title_clean.values,
                            vocabulary, VOCAB_SIZE, MAX_TITLE_LEN, CLASSES_NUM)
test_dataset  = NewsDataset(df_test.target_tags.values,  df_test.title_clean.values,
                            vocabulary, VOCAB_SIZE, MAX_TITLE_LEN, CLASSES_NUM)

train_loader = DataLoader(train_dataset, shuffle=True,  batch_size=64,   num_workers=2)
val_loader   = DataLoader(val_dataset,   shuffle=False, batch_size=4096, num_workers=2)
print('Датасеты готовы')

## 1.2 Базовая модель

Запустите обучение и изучите, как устроена модель.

In [ ]:
class SimpleClassifier(nn.Module):
    """
    Embedding → среднее по словам → Linear → логиты классов.
    Каждое слово заголовка превращается в вектор размера embedding_dim.
    Векторы усредняются в один «вектор смысла» заголовка.
    """
    def __init__(self, vocab_size, embedding_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fc = nn.Linear(embedding_dim, output_dim)

    def forward(self, title):
        embedded = self.embedding(title)   # (B, L, E)
        pooled   = embedded.mean(dim=1)    # (B, E)
        return self.fc(pooled)             # (B, num_classes)


def train_model(model, train_loader, val_loader, num_epochs=5, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()  # multi-label: BCE, не CrossEntropy

    train_losses, val_losses = [], []
    for epoch in range(1, num_epochs + 1):
        model.train()
        total = 0.0
        for X, y in tqdm(train_loader, desc=f'Epoch {epoch}', leave=False):
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(X)
        train_losses.append(total / len(train_loader.dataset))

        model.eval()
        total = 0.0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                total += criterion(model(X), y).item() * len(X)
        val_losses.append(total / len(val_loader.dataset))
        print(f'Epoch {epoch}: train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}')

    return train_losses, val_losses


model_baseline = SimpleClassifier(VOCAB_SIZE, embedding_dim=300, output_dim=CLASSES_NUM)
train_losses, val_losses = train_model(model_baseline, train_loader, val_loader, num_epochs=5)

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Кривая обучения'); plt.tight_layout(); plt.show()

## 1.3 Метрики

Для каждой новости модель предсказывает **несколько тегов** (multi-label). Метрики считаются как пересечение предсказанного и реального множеств тегов.

In [ ]:
def get_predictions(model, dataset):
    """Возвращает вероятности и таргеты для всего датасета."""
    loader = DataLoader(dataset, shuffle=False, batch_size=len(dataset))
    model.eval()
    with torch.no_grad():
        for X, y in loader:
            probs = torch.sigmoid(model(X.to(device))).cpu()
    return probs, y

def precision_recall(target, y_pred):
    tp   = ((y_pred == 1) & (target == 1)).sum(dim=1).float()
    prec = (tp / ((y_pred == 1).sum(dim=1).float() + 1e-5)).mean().item()
    rec  = (tp / ((target == 1).sum(dim=1).float() + 1e-5)).mean().item()
    return prec, rec

### Задание 1 (0.4 балл) — подбор порога

Модель выдаёт вероятность для каждого тега. Чтобы получить предсказание, нужно выбрать **порог**: теги с вероятностью выше порога считаются предсказанными.

**Что нужно сделать:**
1. Переберите пороги от `0.01` до `0.5` с шагом `0.01`
2. На каждом пороге посчитайте precision и recall на `val_dataset`
3. Выберите порог, максимизирующий F1 = 2 · P · R / (P + R)
4. Оцените baseline с этим порогом на `test_dataset`

**Ответьте в тексте:** что происходит с precision и recall при увеличении порога?

In [ ]:
val_probs, val_target = get_predictions(model_baseline, val_dataset)

# YOUR CODE HERE
# Переберите пороги, найдите лучший по F1 на val
best_threshold = ...
print(f'Лучший порог: {best_threshold}')

In [ ]:
# Оценка на тестовой выборке
test_probs, test_target = get_predictions(model_baseline, test_dataset)
y_pred = (test_probs > best_threshold).int()
prec, rec = precision_recall(test_target, y_pred)
f1 = 2 * prec * rec / (prec + rec + 1e-5)
print(f'Test — Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}')

**Ваш ответ:** *[напишите здесь]*

### Задание 2 (0.8 балла) — улучшенная модель

Базовая модель усредняет все слова заголовка одинаково. Можно сделать лучше: добавить **свёрточные слои**, которые умеют улавливать n-граммы (пары и тройки слов).

Реализуйте `CNNClassifier`:
1. `nn.Embedding` — как раньше
2. Несколько `nn.Conv1d` с разными размерами ядра (например, 2 и 3 — биграммы и триграммы)
3. `F.max_pool1d` по всей длине последовательности — берём самый яркий признак
4. Конкатенация выходов всех свёрток → `nn.Linear`

Обучите `CNNClassifier` и сравните его F1 с baseline.

> 💡 **Подсказка по размерностям**: после `Conv1d(in, out, kernel)` на входе `(B, E, L)` получается `(B, out, L-kernel+1)`. После `max_pool1d` — `(B, out, 1)`, после `squeeze` — `(B, out)`.

In [ ]:
class CNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_filters, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # YOUR CODE HERE
        # Добавьте Conv1d с ядрами разных размеров

    def forward(self, title):
        embedded = self.embedding(title).transpose(1, 2)  # (B, E, L)

        # YOUR CODE HERE
        pass


# YOUR CODE HERE
# Обучите CNNClassifier и сравните F1 с baseline

### Задание 3 (0.8 балла) — HuggingFace pipeline для классификации

Вместо обучения с нуля можно взять **предобученный BERT** и добавить к нему классификационную голову. Это называется fine-tuning.

Используйте `cointegrated/rubert-tiny2` — маленький BERT для русского языка (~30MB, обучается за 5–10 минут на Colab T4).

**Что нужно сделать:**
1. Загрузить токенизатор и модель
2. Добавить линейный слой поверх `[CLS]`-токена
3. Обучить (заморозьте BERT или обучайте всё — на ваш выбор)
4. Сравнить F1 с `SimpleClassifier` и `CNNClassifier`

**Ответьте:** почему предобученный BERT может быть лучше ваших моделей? Почему он может оказаться хуже при малом числе данных?

> 💡 **Подсказка**: `outputs = model(input_ids, attention_mask)` — берите `outputs.last_hidden_state[:, 0, :]` (вектор `[CLS]`) для классификации.

Для удобства за вас уже написаны функции обучения и предсказания

In [ ]:
# !pip install transformers -q
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'cointegrated/rubert-tiny2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# YOUR CODE HERE
# 1. Создайте датасет, который токенизирует через tokenizer
# 2. Создайте модель: AutoModel + nn.Linear(312, CLASSES_NUM)
# 3. Обучите и оцените

def train_model(model, train_loader, val_loader, num_epochs=5, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()  # multi-label: BCE, не CrossEntropy

    train_losses, val_losses = [], []
    for epoch in range(1, num_epochs + 1):
        model.train()
        total = 0.0
        for X, y in tqdm(train_loader, desc=f'Epoch {epoch}', leave=False):
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(X)
        train_losses.append(total / len(train_loader.dataset))

        model.eval()
        total = 0.0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                total += criterion(model(X), y).item() * len(X)
        val_losses.append(total / len(val_loader.dataset))
        print(f'Epoch {epoch}: train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}')

    return train_losses, val_losses


def train_bert(model, train_loader, val_loader, num_epochs=3, lr=2e-5):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    train_losses, val_losses = [], []
    for epoch in range(1, num_epochs + 1):
        model.train()
        total = 0.0
        for ids, mask, y in tqdm(train_loader, desc=f'BERT Epoch {epoch}', leave=False):
            ids, mask, y = ids.to(device), mask.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(ids, mask), y)
            loss.backward(); optimizer.step()
            total += loss.item() * len(y)
        train_losses.append(total / len(train_loader.dataset))

        model.eval()
        total = 0.0
        with torch.no_grad():
            for ids, mask, y in val_loader:
                ids, mask, y = ids.to(device), mask.to(device), y.to(device)
                total += criterion(model(ids, mask), y).item() * len(y)
        val_losses.append(total / len(val_loader.dataset))
        print(f'BERT Epoch {epoch}: train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}')
    return train_losses, val_losses

def get_bert_predictions(model, dataset):
    loader = DataLoader(dataset, shuffle=False, batch_size=256)
    model.eval()
    all_probs, all_y = [], []
    with torch.no_grad():
        for ids, mask, y in loader:
            probs = torch.sigmoid(model(ids.to(device), mask.to(device))).cpu()
            all_probs.append(probs); all_y.append(y)
    return torch.cat(all_probs), torch.cat(all_y)

**Ваш ответ:** *[напишите здесь]*

### Итоговое сравнение моделей

In [ ]:
# YOUR CODE HERE
# Выведите сводную таблицу:
# Модель          | Precision | Recall | F1
# SimpleClassifier|           |        |
# CNNClassifier   |           |        |
# BERT fine-tuned |           |        |

---

# Часть 2. Сентимент-анализ комментариев (1 балла)

Используем готовую модель из HuggingFace — нам не нужно ничего обучать.

### Задание 4 (0.25 балла) — изучите модель

Откройте страницу [seara/rubert-tiny2-russian-sentiment](https://huggingface.co/seara/rubert-tiny2-russian-sentiment) и ответьте:

1. Какова архитектура? Сколько параметров?
2. На каких данных обучена модель?
3. Адекватно ли применять её к комментариям ВКонтакте — почему?

**Ваш ответ:**

1. ...
2. ...
3. ...

### Задание 5 (0.75 балла) — запустите классификатор

In [ ]:
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model='seara/rubert-tiny2-russian-sentiment',
    device=0 if torch.cuda.is_available() else -1
)

# Проверка на примерах
examples = ['Отличная новость!', 'Ужасно, всё плохо', 'Ничего особенного']
print(classifier(examples))

In [ ]:
# Прогоняем на выборке из 50 000 комментариев
# (прогон всех 2.6М займёт несколько часов — не нужно)
sample_comments = df_comments.sample(50_000, random_state=42).copy()

# YOUR CODE HERE
# Прогоните classifier на sample_comments['text']
# Сохраните label и score в новые столбцы
# Подсказка: используйте batch_size=128 для ускорения

# sample_comments['sentiment_label'] = ...
# sample_comments['sentiment_score'] = ...

In [ ]:
# Сохраните результат, чтобы не пересчитывать
# sample_comments.to_csv('comments_sentiment.csv', index=False)

---

# Часть 3. Аналитика (1 балла)

### Задание 6 (0.33 балл) — общая картина сентимента

1. Постройте bar chart с долей позитивных / нейтральных / негативных комментариев
2. Выведите 10 самых позитивных (по `sentiment_score`). Негативные не выводите, там грустно... :(

In [ ]:
# YOUR CODE HERE

### Задание 7 (0.33 балл) — сентимент по новостям

Для каждой новости из `df_vk` посчитайте число позитивных и негативных комментариев (через join с `sample_comments`).

Затем ответьте с визуализацией:
- Есть ли связь между числом лайков под новостью и долей позитивных комментариев?
- Что вас удивило?

In [ ]:
# YOUR CODE HERE

**Ваш ответ:** *[напишите здесь]*

### Задание 8 (0.33 балл) — топ позитивных новостей

Найдите топ-10 новостей с наибольшей долей позитивных комментариев.

**Проблема простой сортировки**: если под новостью 1 комментарий и он позитивный — доля 100%, но это случайность. Нужно учитывать число голосов.

Используйте простое правило: **учитывайте только новости, у которых не менее 10 комментариев** в вашей выборке. Среди них отсортируйте по доле позитивных.

Выведите топ-10: заголовок, доля позитивных, число комментариев.

In [ ]:
# YOUR CODE HERE

---

## Итоговый отчёт

1. Какая модель показала лучший F1? Почему, на ваш взгляд?
2. Что вас удивило в результатах аналитики?
